# D1 — Single-agent ReAct foundation
This notebook imports the one implementation in `src/claim_agent.py`; it does not duplicate the agent. The hand-written loop emits **Thought → Action → Observation**, repeats based on fixture evidence, and then emits **Final**. Independent line coverage and hospital calls share an Action block, while policy and pre-authorisation dependencies are ordered. The default scripted backend is offline. The sole simulated state-changing tool is confirmation-gated and writes at most once per run.


In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory
from pprint import pprint
from src.claim_agent import ClaimAgent, BACKEND, MODEL
print(f'backend={BACKEND}, model={MODEL}')


## Successful partly payable case
The claim is approved in principle while its excluded line remains a line-level refusal. Confirmation permits one local record in a temporary directory.


In [ ]:
tmp = TemporaryDirectory()
agent = ClaimAgent(log_path=Path(tmp.name) / 'decisions.jsonl')
success = agent.run('CLM-8842', confirm=True)
pprint(success.trace)
pprint(success.decision_record)
assert success.decision_record['decision'] == 'approve_in_principle'
assert success.write_count == 1


## Negative case and rejected gate
A lapsed policy escalates without unnecessary line or hospital checks. Here confirmation is deliberately absent, so no decision record is appended.


In [ ]:
negative_log = Path(tmp.name) / 'blocked.jsonl'
negative = ClaimAgent(log_path=negative_log).run('CLM-8910', confirm=False)
pprint(negative.trace)
pprint(negative.decision_record)
assert negative.decision_record['trigger'] == 'policy_lapsed'
assert negative.write_count == 0 and not negative_log.exists()
tmp.cleanup()
